In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd()

# Support running the notebook either from project root or model/
if not (PROJECT_DIR / "data" / "raw").exists():
    if (PROJECT_DIR / "model" / "data" / "raw").exists():
        PROJECT_DIR = PROJECT_DIR / "model"

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "output" / "tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANNUAL_FILE = PROCESSED_DIR / "london_underground_clean.csv"

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)

Project directory: /Users/ovo/Desktop/Grad/Internship/Data_for_Humanity_Internship/do-starter-playground/Model
Raw data directory: /Users/ovo/Desktop/Grad/Internship/Data_for_Humanity_Internship/do-starter-playground/Model/data/raw


In [2]:

# ============================================================
# 2. Expected NUMBAT files
# ============================================================

FILE_MAP = {
    (2019, "Mon-Thu"): "NBT19MTT_Outputs",
    (2019, "Friday"):  "NBT19FRI_Outputs",
    (2019, "Saturday"):"NBT19SAT_Outputs",
    (2019, "Sunday"):"NBT19SUN_Outputs",

    (2022, "Monday"):  "NBT22MON_Outputs",
    (2022, "Tue-Thu"): "NBT22TWT_Outputs",
    (2022, "Friday"):  "NBT22FRI_Outputs",
    (2022, "Saturday"):"NBT22SAT_Outputs",
    (2022, "Sunday"):"NBT22SUN_Outputs",

    (2025, "Monday"):  "NBT25MON_Outputs",
    (2025, "Tue-Thu"): "NBT25TWT_Outputs",
    (2025, "Friday"):  "NBT25FRI_Outputs",
    (2025, "Saturday"):"NBT25SAT_Outputs",
    (2025, "Sunday"):"NBT25SUN_Outputs",
}


def find_excel_file(stem):
    """Find a file by its name, ignoring extension case."""
    matches = [
        p for p in RAW_DIR.iterdir()
        if p.is_file() and p.stem.casefold() == stem.casefold()
    ]

    if len(matches) == 0:
        raise FileNotFoundError(f"Missing file: {stem} (.xlsx)")
    if len(matches) > 1:
        raise ValueError(f"Multiple files found for {stem}: {matches}")

    return matches[0]


FILE_PATHS = {
    key: find_excel_file(stem)
    for key, stem in FILE_MAP.items()
}

print(f"\nFound {len(FILE_PATHS)} NUMBAT files:")
for key, path in FILE_PATHS.items():
    print(f"  {key}: {path.name}")




Found 14 NUMBAT files:
  (2019, 'Mon-Thu'): NBT19MTT_Outputs.xlsx
  (2019, 'Friday'): NBT19FRI_Outputs.xlsx
  (2019, 'Saturday'): NBT19SAT_Outputs.xlsx
  (2019, 'Sunday'): NBT19SUN_Outputs.xlsx
  (2022, 'Monday'): NBT22MON_Outputs.xlsx
  (2022, 'Tue-Thu'): NBT22TWT_Outputs.xlsx
  (2022, 'Friday'): NBT22FRI_Outputs.xlsx
  (2022, 'Saturday'): NBT22SAT_Outputs.xlsx
  (2022, 'Sunday'): NBT22SUN_Outputs.xlsx
  (2025, 'Monday'): NBT25MON_Outputs.xlsx
  (2025, 'Tue-Thu'): NBT25TWT_Outputs.xlsx
  (2025, 'Friday'): NBT25FRI_Outputs.xlsx
  (2025, 'Saturday'): NBT25SAT_Outputs.xlsx
  (2025, 'Sunday'): NBT25SUN_Outputs.xlsx


In [3]:

# ============================================================
# 3. Load London Underground station reference
# ============================================================

annual_df = pd.read_csv(ANNUAL_FILE)

annual_df.columns = annual_df.columns.astype(str).str.strip()

annual_df["Mode"] = annual_df["Mode"].astype(str).str.strip()
annual_df["MASC"] = annual_df["MASC"].astype(str).str.strip()

lu_annual = annual_df[
    annual_df["Mode"].str.upper().eq("LU")
].copy()

# Use the latest available station name as the canonical name
station_map = (
    lu_annual.sort_values("Year", ascending=False)
    .dropna(subset=["MASC"])
    .drop_duplicates(subset=["MASC"])
    [["MASC", "Station"]]
    .rename(columns={"MASC": "ASC"})
)

station_map["ASC"] = station_map["ASC"].astype(str).str.strip()

lu_asc_codes = set(station_map["ASC"])

print("\nLU station codes in Annual Counts:", len(lu_asc_codes))



LU station codes in Annual Counts: 269


In [4]:


# ============================================================
# 4. Identify the 96 quarter-hour columns
# ============================================================

def get_qhr_columns(df):
    """Find columns formatted like 0500-0515."""
    return [
        col for col in df.columns
        if re.fullmatch(r"\d{4}-\d{4}", str(col).strip())
    ]




In [5]:

# ============================================================
# 5. Read and combine Station Entries + Station Exits
# ============================================================

def get_sheet_name(excel_file, target_name):
    """Find a worksheet name without relying on capitalization."""
    sheet_lookup = {
        name.strip().casefold(): name
        for name in excel_file.sheet_names
    }

    key = target_name.casefold()

    if key not in sheet_lookup:
        raise ValueError(
            f"Worksheet '{target_name}' not found in "
            f"{excel_file.io}. Available sheets: {excel_file.sheet_names}"
        )

    return sheet_lookup[key]


def read_station_sheet(excel_file, sheet_name):
    df = pd.read_excel(
    excel_file,
    sheet_name=sheet_name,
    header=2
)

    df.columns = df.columns.astype(str).str.strip()

    # Standardize ASC values
    if "ASC" not in df.columns:
        raise ValueError(
            f"ASC column missing in {sheet_name}. "
            f"Columns found: {df.columns.tolist()}"
        )

    df["ASC"] = df["ASC"].astype(str).str.strip()

    qhr_cols = get_qhr_columns(df)

    if len(qhr_cols) != 96:
        raise ValueError(
            f"Expected 96 quarter-hour columns in {sheet_name}, "
            f"but found {len(qhr_cols)}."
        )

    # Convert passenger counts to numeric
    for col in qhr_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Aggregate in case a station appears more than once
    # in the same sheet
    df = (
        df.groupby("ASC", as_index=False)[qhr_cols]
        .sum(min_count=1)
    )

    return df, qhr_cols


def process_numbat_file(file_path):
    """Combine station entries and exits into total passenger flow."""

    excel_file = pd.ExcelFile(file_path)

    entries, qhr_cols = read_station_sheet(
        excel_file, "Station_Entries"
    )

    exits, exit_qhr_cols = read_station_sheet(
        excel_file, "Station_Exits"
    )

    if qhr_cols != exit_qhr_cols:
        raise ValueError(
            f"Quarter-hour columns differ between Entries and Exits: "
            f"{file_path.name}"
        )

    # Check that both sheets contain the same station codes
    entry_codes = set(entries["ASC"])
    exit_codes = set(exits["ASC"])

    if entry_codes != exit_codes:
        only_entries = sorted(entry_codes - exit_codes)
        only_exits = sorted(exit_codes - entry_codes)

        raise ValueError(
            f"Station mismatch in {file_path.name}\n"
            f"Only in Entries: {only_entries[:10]}\n"
            f"Only in Exits: {only_exits[:10]}"
        )

    # Merge entries and exits
    flow = entries.merge(
        exits,
        on="ASC",
        how="inner",
        suffixes=("_Entry", "_Exit"),
        validate="one_to_one"
    )

    # Total flow = entries + exits
    for col in qhr_cols:
        flow[col] = (
            flow[f"{col}_Entry"] + flow[f"{col}_Exit"]
        )

    flow = flow[["ASC"] + qhr_cols].copy()

    return flow, qhr_cols



In [6]:

# ============================================================
# 6. Process all 11 files
# ============================================================

processed = {}
all_qhr_cols = None

for (year, day_type), file_path in FILE_PATHS.items():

    print(f"\nProcessing {year} {day_type}: {file_path.name}")

    flow, qhr_cols = process_numbat_file(file_path)

    if all_qhr_cols is None:
        all_qhr_cols = qhr_cols
    elif all_qhr_cols != qhr_cols:
        raise ValueError(
            f"Quarter-hour columns differ in {file_path.name}"
        )

    # Keep only LU stations using Annual Counts MASC codes
    flow = flow[flow["ASC"].isin(lu_asc_codes)].copy()

    # Add canonical station names
    flow = flow.merge(
        station_map,
        on="ASC",
        how="left",
        validate="one_to_one"
    )

    flow.insert(0, "Year", year)
    flow.insert(1, "Day_Type", day_type)

    processed[(year, day_type)] = flow

    print(
        f"  LU stations: {len(flow)} | "
        f"Total flow across all bins: "
        f"{flow[qhr_cols].sum().sum():,.0f}"
    )



Processing 2019 Mon-Thu: NBT19MTT_Outputs.xlsx
  LU stations: 267 | Total flow across all bins: 9,523,707

Processing 2019 Friday: NBT19FRI_Outputs.xlsx
  LU stations: 267 | Total flow across all bins: 9,791,556

Processing 2019 Saturday: NBT19SAT_Outputs.xlsx
  LU stations: 267 | Total flow across all bins: 7,143,764

Processing 2019 Sunday: NBT19SUN_Outputs.xlsx
  LU stations: 267 | Total flow across all bins: 4,896,190

Processing 2022 Monday: NBT22MON_Outputs.xlsx
  LU stations: 269 | Total flow across all bins: 6,700,127

Processing 2022 Tue-Thu: NBT22TWT_Outputs.xlsx
  LU stations: 269 | Total flow across all bins: 7,744,830

Processing 2022 Friday: NBT22FRI_Outputs.xlsx
  LU stations: 269 | Total flow across all bins: 7,246,032

Processing 2022 Saturday: NBT22SAT_Outputs.xlsx
  LU stations: 269 | Total flow across all bins: 6,524,803

Processing 2022 Sunday: NBT22SUN_Outputs.xlsx
  LU stations: 269 | Total flow across all bins: 4,515,718

Processing 2025 Monday: NBT25MON_Output

In [7]:

# ============================================================
# 7. Create comparable Mon-Thu profiles
# ============================================================

weekday_profiles = []

# 2019: use MTT directly
weekday_2019 = processed[(2019, "Mon-Thu")].copy()
weekday_profiles.append(weekday_2019)

# 2022 and 2025: Monday + 3 * Tue-Thu, divided by 4
for year in [2022, 2025]:

    monday = processed[(year, "Monday")].copy()
    tue_thu = processed[(year, "Tue-Thu")].copy()

    # Match stations before calculating the weighted average
    monday = monday.set_index("ASC").sort_index()
    tue_thu = tue_thu.set_index("ASC").sort_index()

    if set(monday.index) != set(tue_thu.index):
        raise ValueError(
            f"Station codes differ between Monday and Tue-Thu in {year}."
        )

    combined = monday.copy()

    combined[all_qhr_cols] = (
        monday[all_qhr_cols] + 3 * tue_thu[all_qhr_cols]
    ) / 4

    combined = combined.reset_index()
    combined["Year"] = year
    combined["Day_Type"] = "Mon-Thu"

    weekday_profiles.append(
        combined[["Year", "Day_Type", "ASC", "Station"] + all_qhr_cols]
    )



In [8]:

# ============================================================
# 8. Combine all day types into one dataset
# ============================================================

other_profiles = []

for key, df in processed.items():
    year, day_type = key

    # Monday and Tue-Thu are used only to construct Mon-Thu
    if day_type in ["Monday", "Tue-Thu", "Mon-Thu"]:
        continue

    other_profiles.append(df)


final_df = pd.concat(
    weekday_profiles + other_profiles,
    ignore_index=True
)

final_df = final_df[
    ["Year", "Day_Type", "ASC", "Station"] + all_qhr_cols
].copy()

final_df = final_df.sort_values(
    ["Year", "Day_Type", "Station"]
).reset_index(drop=True)



In [9]:

# ============================================================
# 9. Add daily total flow and save
# ============================================================

final_df["Daily_Flow"] = final_df[all_qhr_cols].sum(axis=1)

output_file = PROCESSED_DIR / "numbat_station_profiles.csv"

final_df.to_csv(output_file, index=False)

print("\n========================================")
print("PREPROCESSING COMPLETE")
print("========================================")
print("Output:", output_file)
print("Dataset shape:", final_df.shape)

print("\nRows per year and day type:")
print(
    final_df.groupby(["Year", "Day_Type"])
    .size()
    .unstack(fill_value=0)
)

print("\nUnique stations per year:")
print(
    final_df.groupby("Year")["ASC"].nunique()
)

print("\nMissing values in quarter-hour columns:")
print(final_df[all_qhr_cols].isna().sum().sum())

print("\nPreview:")
display(final_df.head())


PREPROCESSING COMPLETE
Output: /Users/ovo/Desktop/Grad/Internship/Data_for_Humanity_Internship/do-starter-playground/Model/data/processed/numbat_station_profiles.csv
Dataset shape: (3220, 101)

Rows per year and day type:
Day_Type  Friday  Mon-Thu  Saturday  Sunday
Year                                       
2019         267      267       267     267
2022         269      269       269     269
2025         269      269       269     269

Unique stations per year:
Year
2019    267
2022    269
2025    269
Name: ASC, dtype: int64

Missing values in quarter-hour columns:
0

Preview:


,Year,Day_Type,ASC,Station,0500-0515,0515-0530,0530-0545,0545-0600,0600-0615,0615-0630,...,0245-0300,0300-0315,0315-0330,0330-0345,0345-0400,0400-0415,0415-0430,0430-0445,0445-0500,Daily_Flow
0,2019,Friday,ACTu,Acton Town,16.7320,32.2792,53.0082,81.8154,110.7444,147.5696,...,30.069,26.4088,29.1808,26.8166,29.3548,26.4702,32.5634,24.9454,21.0824,20819.588804
1,2019,Friday,ALDu,Aldgate,7.0292,11.8854,27.8046,39.2266,68.8404,102.2734,...,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,32330.824409
2,2019,Friday,ALEu,Aldgate East,12.5400,25.8030,41.7298,60.4902,83.2734,109.1024,...,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,48440.550213
3,2019,Friday,ALPu,Alperton,16.7546,35.0548,52.4168,79.9078,103.3752,135.8972,...,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,9707.210593
4,2019,Friday,AMEu,Amersham,16.8454,16.9278,32.2344,50.2592,66.8472,80.1792,...,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,7503.996998
